# Legacy Replication: Wisconsin County-Level PM2.5 Spatial Analysis (2024)

This notebook documents the legacy replication layer of the rebuild. Because the original raw EPA daily export and the original course notebook were not available in the workspace, the county-level benchmark snapshot is used as a transparent fallback.

In [1]:
from pathlib import Path
import sys
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import PM25_FIELD

county_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "county_annual_pm25.csv")
coverage_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "county_monitor_coverage.csv")
joined_gdf = gpd.read_file(PROJECT_ROOT / "outputs" / "tables" / "wisconsin_counties_pm25_joined.geojson")
spatial_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "spatial_sensitivity_results.csv")
weights_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "spatial_weights_comparison.csv")
stability_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "local_cluster_stability.csv")
pm25_population_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "pm25_population_context.csv")


## Legacy county snapshot

In [2]:
print(county_df[['County_FIPS_Code', 'County', PM25_FIELD, 'n_sites', 'Data_Mode']].head(16).to_string(index=False))

 County_FIPS_Code     County  PM25_Annual_Mean_2024  n_sites                Data_Mode
            55003    Ashland               4.697527        1 fallback_county_snapshot
            55009      Brown               6.135246        1 fallback_county_snapshot
            55025       Dane               6.434959        2 fallback_county_snapshot
            55027      Dodge               5.394867        1 fallback_county_snapshot
            55035 Eau Claire               6.448328        1 fallback_county_snapshot
            55041     Forest               4.722958        1 fallback_county_snapshot
            55043      Grant               7.346175        1 fallback_county_snapshot
            55053    Jackson               5.677617        1 fallback_county_snapshot
            55059    Kenosha               5.972404        1 fallback_county_snapshot
            55073   Marathon               5.492896        1 fallback_county_snapshot
            55079  Milwaukee               7.037637   

## Monitor coverage

In [3]:
print(coverage_df[['has_monitor']].value_counts().rename('count').to_string())

has_monitor
False          56
True           16


## Legacy benchmark range

In [4]:
print(county_df.sort_values(PM25_FIELD, ascending=False)[['County', PM25_FIELD, 'n_sites']].to_string(index=False))

    County  PM25_Annual_Mean_2024  n_sites
     Grant               7.346175        1
 Milwaukee               7.037637        2
  Waukesha               6.970588        1
Eau Claire               6.448328        1
      Dane               6.434959        2
     Brown               6.135246        1
 Outagamie               6.106887        1
   Kenosha               5.972404        1
   Jackson               5.677617        1
      Sauk               5.600820        1
  Marathon               5.492896        1
     Dodge               5.394867        1
   Ozaukee               5.109836        1
    Forest               4.722958        1
   Ashland               4.697527        1
    Monroe               4.546407        1


## Legacy figures

![Choropleth](../outputs/figures/pm25_2024_choropleth.png)

![Ranking](../outputs/figures/pm25_2024_ranked_bar.png)

![Legacy LISA](../outputs/figures/lisa_cluster_map_legacy.png)

## Spatial diagnostics

In [5]:
legacy = spatial_df[spatial_df['specification'] == 'legacy_queen']
print(legacy[['County', 'cluster', 'local_moran_I', 'local_p_sim', 'is_island']].sort_values('County').to_string(index=False))

    County         cluster  local_moran_I  local_p_sim  is_island
   Ashland    No neighbors      -0.000000        0.001       True
     Brown Not significant       0.095238        0.398      False
      Dane Not significant      -0.281752        0.332      False
     Dodge Not significant      -0.530568        0.094      False
Eau Claire Not significant      -0.143548        0.471      False
    Forest    No neighbors      -0.000000        0.001       True
     Grant    No neighbors       0.000000        0.001       True
   Jackson Not significant       0.086892        0.292      False
   Kenosha    No neighbors       0.000000        0.001       True
  Marathon    No neighbors      -0.000000        0.001       True
 Milwaukee Not significant       0.295881        0.287      False
    Monroe Not significant       0.317332        0.453      False
 Outagamie Not significant       0.095238        0.408      False
   Ozaukee        Low-High      -1.164068        0.024      False
      Sauk

Legacy Queen weights contain disconnected monitored counties. Island counties in this rebuild are: Ashland, Forest, Grant, Kenosha, Marathon.